## Ragas evaluation — all QA approaches vs gold answers

Scores predictions from `qa_eval_*_outputs.jsonl` against `gold_answer` in `qa_eval_dataset.jsonl` using [Ragas](https://docs.ragas.io/):

Can be also set to one key in `APPROACH_REGISTRY` or `RUN_ALL_APPROACHES = True` to score every system.

Requires batch prediction files from the QA notebooks and an OpenAI API key (`OPENAI_API_KEY`) for the Ragas LLM judge.

In [26]:
import importlib.util
import subprocess
import sys

INSTALL_ARGS = [
    'ragas>=0.2.0',
    'datasets',
    'langchain-community',
    'sentence-transformers',
]

need_install = importlib.util.find_spec('ragas') is None
need_install = need_install or importlib.util.find_spec('sentence_transformers') is None

if need_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *INSTALL_ARGS])
else:
    print('ragas + sentence-transformers ready')

ragas + sentence-transformers ready


In [ ]:
from pathlib import Path
import json
import os
import re
import warnings

import requests
from datasets import Dataset
from openai import OpenAI
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import llm_factory
from ragas.metrics import answer_correctness, answer_relevancy, faithfulness
from langchain_community.embeddings import HuggingFaceEmbeddings

warnings.filterwarnings('ignore', category=DeprecationWarning, module='ragas')

project_root = Path('.').resolve()
gold_path = project_root / 'qa_eval_dataset.jsonl'

APPROACH = 'graphdb'  # graphdb | rag_with_events | rag_no_events | hybrid | cdf_json_rag
RUN_ALL_APPROACHES = False

APPROACH_REGISTRY = {
    'graphdb': 'qa_eval_graphdb_outputs.jsonl',
    'rag_with_events': 'qa_eval_rag_with_events_outputs.jsonl',
    'rag_no_events': 'qa_eval_rag_no_events_outputs.jsonl',
    'hybrid': 'qa_eval_hybrid_outputs.jsonl',
    'cdf_json_rag': 'qa_eval_cdf_json_rag_outputs.jsonl',
}

QUESTION_LIMIT = None

# LLM judge (OpenAI API — used only by Ragas, not the QA notebooks)
LLM_API_KEY = os.getenv('OPENAI_API_KEY') or os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-5.4-mini')
if not LLM_API_KEY:
    raise ValueError('Set OPENAI_API_KEY in your environment before running Ragas.')

# GraphDB fallback for legacy outputs missing contexts
SPARQL_ENDPOINT = os.getenv(
    'SPARQL_ENDPOINT',
    'http://localhost:7200/repositories/Master_Thesis',
)
ENABLE_SPARQL_CONTEXT_FALLBACK = True

# Local embeddings for answer_relevancy
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
RAGAS_BATCH_SIZE = 8

print('Gold:', gold_path.exists())
print('Approaches:', list(APPROACH_REGISTRY.keys()))
for key, fname in APPROACH_REGISTRY.items():
    exists = (project_root / fname).exists()
    print(f'  {key}: {fname} ({"found" if exists else "missing"})')

Gold: True
Approaches: ['graphdb', 'rag_with_events', 'rag_no_events', 'hybrid', 'cdf_json_rag']
  graphdb: qa_eval_graphdb_outputs.jsonl (found)
  rag_with_events: qa_eval_rag_with_events_outputs.jsonl (found)
  rag_no_events: qa_eval_rag_no_events_outputs.jsonl (found)
  hybrid: qa_eval_hybrid_outputs.jsonl (found)
  cdf_json_rag: qa_eval_cdf_json_rag_outputs.jsonl (found)


C:\Users\dyury\AppData\Local\Temp\ipykernel_9632\2621123063.py:13: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness, answer_relevancy, faithfulness
C:\Users\dyury\AppData\Local\Temp\ipykernel_9632\2621123063.py:13: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import answer_correctness, answer_relevancy, faithfulness
C:\Users\dyury\AppData\Local\Temp\ipykernel_9632\2621123063.py:13: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collecti

In [28]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def normalize_text(text: str) -> str:
    text = (text or '').strip()
    return re.sub(r'\s+', ' ', text)


def normalize_yes_no(text: str) -> str:
    t = normalize_text(text).rstrip('.').lower()
    if t in {'yes', 'no'}:
        return t.capitalize()
    return normalize_text(text)


def format_bindings(result_json: dict, max_rows: int = 30) -> str:
    if result_json.get('boolean') is not None:
        return f'ASK result: {result_json["boolean"]}'
    head_vars = result_json.get('head', {}).get('vars', [])
    bindings = result_json.get('results', {}).get('bindings', [])
    if not bindings:
        return 'No rows returned.'
    lines = []
    for i, row in enumerate(bindings[:max_rows], start=1):
        vals = [f"{v}={row.get(v, {}).get('value', '')}" for v in head_vars]
        lines.append(f'{i}. ' + '; '.join(vals))
    if len(bindings) > max_rows:
        lines.append(f'... ({len(bindings) - max_rows} more rows)')
    return '\n'.join(lines)


def run_sparql(query: str, endpoint: str = SPARQL_ENDPOINT) -> dict:
    resp = requests.post(
        endpoint,
        data=query,
        headers={'Accept': 'application/sparql-results+json', 'Content-Type': 'application/sparql-query'},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()


def contexts_from_prediction(pred: dict, approach: str) -> list[str]:
    raw = pred.get('contexts')
    if isinstance(raw, list) and any(str(c).strip() for c in raw):
        return [str(c) for c in raw if str(c).strip()]
    # Legacy hybrid/graphdb rows routed to Chroma without stored contexts
    if approach in ('graphdb', 'hybrid') and pred.get('route') == 'chroma':
        chunks = pred.get('chunks') or []
        return [str(c) for c in chunks if str(c).strip()]
    # Legacy graphdb/hybrid graphdb-route rows without contexts field
    if ENABLE_SPARQL_CONTEXT_FALLBACK and pred.get('sparql'):
        try:
            rj = run_sparql(pred['sparql'])
            text = format_bindings(rj)
            return [text] if text else []
        except Exception as e:
            print(f"  SPARQL fallback failed for q={pred.get('question_index')}: {e}")
    return []


def build_aligned_pairs(approach: str, pred_path: Path, question_limit: int | None = None) -> list[dict]:
    gold_rows = load_jsonl(gold_path)
    if question_limit is not None:
        gold_rows = gold_rows[:question_limit]

    pred_rows = load_jsonl(pred_path)
    pred_by_question = {row['question']: row for row in pred_rows}
    pred_by_index = {
        int(row['question_index']): row
        for row in pred_rows
        if row.get('question_index') is not None
    }

    pairs: list[dict] = []
    missing_predictions = []

    for i, gold in enumerate(gold_rows, start=1):
        pred = pred_by_question.get(gold['question']) or pred_by_index.get(i)
        if pred is None:
            missing_predictions.append(gold.get('id', f'row_{i}'))
            continue

        ref = gold.get('gold_answer', '')
        if gold.get('answer_type') == 'yes_no':
            ref = normalize_yes_no(ref)
        resp = pred.get('answer') or ''
        if gold.get('answer_type') == 'yes_no':
            resp = normalize_yes_no(resp)

        pairs.append({
            'question_index': pred.get('question_index', i),
            'id': gold.get('id'),
            'template_id': gold.get('template_id'),
            'category': gold.get('category'),
            'answer_type': gold.get('answer_type'),
            'scope': gold.get('scope'),
            'question': gold['question'],
            'user_input': gold['question'],
            'response': normalize_text(resp),
            'reference': normalize_text(ref),
            'retrieved_contexts': contexts_from_prediction(pred, approach),
            'approach': approach,
            'route': pred.get('route'),
            'qa_error': pred.get('error'),
        })

    if missing_predictions:
        print(f'  Missing predictions: {len(missing_predictions)}')
    return pairs

In [29]:
def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def summarize_scores(rows: list[dict], key: str) -> dict:
    vals = [r[key] for r in rows if r.get(key) is not None]
    return {'count': len(vals), f'mean_{key}': mean(vals)}


METRIC_COLS = ['answer_correctness', 'answer_relevancy', 'faithfulness']


def uses_max_completion_tokens(model: str) -> bool:
    """OpenAI GPT-5.x dotted ids (e.g. gpt-5.4-mini) need max_completion_tokens."""
    m = model.lower()
    if m.startswith(('gpt-5.', 'gpt-6.', 'gpt-7.', 'gpt-8.', 'gpt-9.')):
        return True
    if len(m) >= 2 and m[0] == 'o' and m[1] in '123456789' and (len(m) == 2 or m[2] in '-_'):
        return True
    if m.startswith('gpt-'):
        ver = m[4:].split('-')[0].split('_')[0]
        try:
            return int(ver) >= 5
        except ValueError:
            return False
    return m == 'codex-mini'


def configure_openai_ragas_llm(llm, model: str) -> None:
    """Ragas 0.4.x misses dotted GPT-5.x model ids when mapping token params."""
    if not uses_max_completion_tokens(model):
        return
    if 'max_tokens' in llm.model_args:
        llm.model_args['max_completion_tokens'] = llm.model_args.pop('max_tokens')
    llm.model_args['temperature'] = 1.0
    llm.model_args.pop('top_p', None)


def attach_ragas_scores(pairs: list[dict], df) -> None:
    if len(df) != len(pairs):
        raise ValueError(f'Ragas returned {len(df)} rows but expected {len(pairs)}')
    for i, pair in enumerate(pairs):
        row = df.iloc[i]
        for col in METRIC_COLS:
            if col in row.index:
                val = row[col]
                pair[col] = float(val) if val is not None else None


def run_ragas_for_approach(approach: str, ragas_llm, ragas_embeddings) -> dict:
    pred_file = APPROACH_REGISTRY[approach]
    pred_path = project_root / pred_file
    output_path = project_root / f'qa_eval_ragas_{approach}_results.jsonl'
    summary_path = project_root / f'qa_eval_ragas_{approach}_summary.json'

    print(f'\n=== Ragas: {approach} ===')
    print('Predictions:', pred_path)
    if not pred_path.exists():
        print('SKIP — prediction file missing')
        return {'approach': approach, 'skipped': True, 'reason': 'missing predictions'}

    pairs = build_aligned_pairs(approach, pred_path, question_limit=QUESTION_LIMIT)
    if not pairs:
        print('SKIP — no aligned pairs')
        return {'approach': approach, 'skipped': True, 'reason': 'no pairs'}

    qa_errors = sum(1 for p in pairs if p.get('qa_error'))
    empty_ctx = sum(1 for p in pairs if not p.get('retrieved_contexts'))
    empty_resp = sum(1 for p in pairs if not p.get('response'))
    print(
        f'Aligned: {len(pairs)} | QA errors: {qa_errors} | '
        f'empty contexts: {empty_ctx} | empty responses: {empty_resp}'
    )

    dataset = Dataset.from_dict({
        'user_input': [p['user_input'] for p in pairs],
        'response': [p['response'] for p in pairs],
        'reference': [p['reference'] for p in pairs],
        'retrieved_contexts': [p['retrieved_contexts'] for p in pairs],
    })

    print('Running Ragas evaluate (3 metrics)...')
    result = evaluate(
        dataset,
        metrics=[answer_correctness, answer_relevancy, faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
        batch_size=RAGAS_BATCH_SIZE,
    )

    df = result.to_pandas()
    attach_ragas_scores(pairs, df)

    summary = {
        'approach': approach,
        'pred_file': pred_file,
        'num_pairs': len(pairs),
        'qa_errors': qa_errors,
        'empty_contexts': empty_ctx,
        'empty_responses': empty_resp,
        'question_limit': QUESTION_LIMIT,
        'llm_model': LLM_MODEL,
        'embed_model': EMBED_MODEL_NAME,
    }
    for col in METRIC_COLS:
        summary[col] = summarize_scores(pairs, col)

    by_answer_type: dict[str, dict] = {}
    for at in sorted({p.get('answer_type') for p in pairs if p.get('answer_type')}):
        subset = [p for p in pairs if p.get('answer_type') == at]
        by_answer_type[at] = {col: summarize_scores(subset, col) for col in METRIC_COLS}
    summary['by_answer_type'] = by_answer_type

    ok_pairs = [p for p in pairs if not p.get('qa_error')]
    summary['without_qa_errors'] = {col: summarize_scores(ok_pairs, col) for col in METRIC_COLS}

    with output_path.open('w', encoding='utf-8') as f:
        for row in pairs:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    print('Overall means:')
    for col in METRIC_COLS:
        print(f'  {col}: {summary[col][f"mean_{col}"]:.4f} (n={summary[col]["count"]})')
    print(f'Wrote {output_path.name} and {summary_path.name}')
    return summary

In [30]:
llm_client = OpenAI(api_key=LLM_API_KEY, timeout=180.0)
ragas_llm = llm_factory(
    LLM_MODEL,
    provider='openai',
    client=llm_client,
    max_tokens=4096,
)
configure_openai_ragas_llm(ragas_llm, LLM_MODEL)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBED_MODEL_NAME)
)
print('Ragas LLM provider: OpenAI')
print('Ragas LLM model:', LLM_MODEL)
print('Uses max_completion_tokens:', uses_max_completion_tokens(LLM_MODEL))
print('Ragas embeddings:', EMBED_MODEL_NAME)

Ragas LLM provider: OpenAI
Ragas LLM model: gpt-5.4-mini
Uses max_completion_tokens: True
Ragas embeddings: sentence-transformers/all-MiniLM-L6-v2


C:\Users\dyury\AppData\Local\Temp\ipykernel_9632\1426669762.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [31]:
approaches_to_run = list(APPROACH_REGISTRY.keys()) if RUN_ALL_APPROACHES else [APPROACH]
all_summaries: list[dict] = []

for approach in approaches_to_run:
    summary = run_ragas_for_approach(approach, ragas_llm, ragas_embeddings)
    all_summaries.append(summary)

if RUN_ALL_APPROACHES:
    compare_path = project_root / 'qa_eval_ragas_all_approaches_summary.json'
    compare_path.write_text(json.dumps(all_summaries, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'\nWrote cross-approach summary to {compare_path.name}')
    print('\nComparison:')
    for s in all_summaries:
        if s.get('skipped'):
            print(f"  {s['approach']}: SKIPPED ({s.get('reason')})")
        else:
            ac = s.get('answer_correctness', {}).get('mean_answer_correctness', 0)
            ar = s.get('answer_relevancy', {}).get('mean_answer_relevancy', 0)
            ff = s.get('faithfulness', {}).get('mean_faithfulness', 0)
            print(
                f"  {s['approach']}: correctness={ac:.4f} "
                f"relevancy={ar:.4f} faithfulness={ff:.4f} (n={s['num_pairs']})"
            )


=== Ragas: graphdb ===
Predictions: C:\Users\dyury\Desktop\Master Thesis\qa_eval_graphdb_outputs.jsonl
Aligned: 1 | QA errors: 0 | empty contexts: 0 | empty responses: 0
Running Ragas evaluate (3 metrics)...


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Batch 1/1:   0%|          | 0/3 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Overall means:
  answer_correctness: 1.0000 (n=1)
  answer_relevancy: 0.0000 (n=1)
  faithfulness: 0.0000 (n=1)
Wrote qa_eval_ragas_graphdb_results.jsonl and qa_eval_ragas_graphdb_summary.json


In [21]:
# Lowest answer_correctness examples (last approach scored, or APPROACH if single-run)
inspect_approach = approaches_to_run[-1] if 'approaches_to_run' in dir() else APPROACH
results_path = project_root / f'qa_eval_ragas_{inspect_approach}_results.jsonl'
if results_path.exists():
    rows = load_jsonl(results_path)
    worst = sorted(rows, key=lambda r: r.get('answer_correctness') or 0)[:8]
    print(f'Worst answer_correctness — {inspect_approach}')
    for row in worst:
        print(
            f"\ncorrectness={row.get('answer_correctness')} | {row.get('id')} | "
            f"error={bool(row.get('qa_error'))} route={row.get('route')}"
        )
        print('Q:', row['question'][:120])
        print('Gold:', row['reference'][:140])
        print('Pred:', row['response'][:140])
else:
    print(f'No results file for {inspect_approach} — run scoring cell above first.')

Worst answer_correctness — cdf_json_rag

correctness=nan | N-YN-1_01 | error=False route=None
Q: Was Josip Stanišić in the starting lineup for Bayer Leverkusen vs FC Köln on 2023-10-08?
Gold: Yes
Pred: I don’t know.

correctness=nan | N-YN-1_02 | error=False route=None
Q: Was Tomáš Čvančara in the starting lineup for Borussia Mönchengladbach vs Bayer Leverkusen on 2023-08-26?
Gold: Yes
Pred: I don’t know.

correctness=nan | N-YN-1_03 | error=False route=None
Q: Was Maximilian Bauer in the starting lineup for Augsburg vs Bayer Leverkusen on 2024-01-13?
Gold: No
Pred: I don't know.
